In [ ]:
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd
import time

# LSQ endpoint
endpoint_url = "http://lsq.aksw.org/sparql"

# Query template (note the double curly braces {{ }})
query_template = """
PREFIX lsqv: <http://lsq.aksw.org/vocab#> 
PREFIX sd: <http://www.w3.org/ns/sparql-service-description#> 

SELECT DISTINCT ?text
WHERE {{ 
   ?query lsqv:hasRemoteExec/lsqv:endpoint <https://dbpedia.org/sparql> .
   ?query lsqv:text ?text . 
   ?query lsqv:hasSpin ?spin .
   ?spin a <http://spinrdf.org/sp#Select> . 
   ?query lsqv:hasStructuralFeatures ?features . 

   FILTER NOT EXISTS {{ ?features lsqv:usesFeature lsqv:Count . }}

   FILTER(!CONTAINS(LCASE(STR(?text)), "dbpo:wikipageid"))
}}
LIMIT 1000
OFFSET {offset}
"""

def run_query(offset):
    sparql = SPARQLWrapper(endpoint_url)
    sparql.setQuery(query_template.format(offset=offset))
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()
    
    rows = []
    for result in results["results"]["bindings"]:
        rows.append(result["text"]["value"])
    return rows

def main():
    all_results = []

    # Run for 100 iterations, increasing OFFSET by 1000 each time
    for i in range(100):
        offset = i * 1000
        print(f"Fetching batch {i+1} with OFFSET {offset}...")
        try:
            rows = run_query(offset)
            all_results.extend(rows)
        except Exception as e:
            print(f"Error at offset {offset}: {e}")
        time.sleep(0.5)  # polite delay so we don’t overload the LSQ server

    # Save results into a CSV
    df = pd.DataFrame(all_results, columns=["text"])
    df.to_csv("sparql_results.csv", index=False, encoding="utf-8")
    print("Results saved to sparql_results.csv")

if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd

df = pd.read_csv("sparql_results.csv")

total_rows = len(df)
unique_rows = df["text"].nunique()

# Number of duplicates
duplicate_count = total_rows - unique_rows

print(f"Total rows: {total_rows}")
print(f"Unique rows: {unique_rows}")
print(f"Duplicate rows: {duplicate_count}")

duplicates = df[df.duplicated("text", keep=False)].sort_values("text")
print("\nSample duplicates:")
print(duplicates.head(10))


In [ ]:
import requests

# ---- Query DBpedia SPARQL endpoint ----
def query_dbpedia(sparql_query):
    """
    Sends a SPARQL query to the DBpedia endpoint and returns the JSON result.
    Returns None if the request fails.
    """
    endpoint = "https://dbpedia.data.dice-research.org/sparql"
    headers = {"Accept": "application/sparql-results+json"}
    params = {"query": sparql_query}
    try:
        response = requests.get(endpoint, headers=headers, params=params)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException:
        return None

In [ ]:
# ---- Load queries from CSV ----
df = pd.read_csv("sparql_results.csv")
print(f"Total queries loaded: {len(df)}")


counter = 0
failed = 0
successful_queries = []

for index, row in df.iterrows():
    sparql_query = row.get("text")
    if sparql_query:
        results = query_dbpedia(sparql_query)
        if results and "results" in results and results["results"]["bindings"]:
            successful_queries.append({
                "sparql_query": sparql_query,
                "results": results["results"]["bindings"]
            })
            counter += 1
        else:
            failed += 1

print(f"Successful queries: {len(successful_queries)}")
print(f"Failed queries: {failed}")

In [ ]:
import json
# ---- Save successful queries to JSON ----
with open("successful_queries.json", "w", encoding="utf-8") as f:
    json.dump(successful_queries, f, ensure_ascii=False, indent=2)

In [ ]:
import json
with open("successful_queries.json", "r") as f:
    data = json.load(f)

# ---- Extract result strings for similarity comparison ----
def extract_result_strings(entry):
    """
    Extracts unique result strings from a query result entry.
    """
    values = set()
    for result in entry.get("results", []):
        for k, v in result.items():
            if isinstance(v, dict) and 'value' in v:
                values.add(f"{k}:{v['value']}")
    return values



preprocessed = [(i, entry['sparql_query'], extract_result_strings(entry)) for i, entry in enumerate(data)]

In [ ]:
def process_query(args, max_similars):
    """
    Finds similar queries based on Jaccard similarity of result sets.
    Returns a dictionary with base query and its similar queries.

    Args:
        args: Tuple containing (base_id, base_query, base_results)
        max_similars: Maximum number of similar queries to return
    """
    base_id, base_query, base_results = args
    local_similars = []
    for sim_id, sim_query, sim_results in preprocessed[base_id + 1:]:
        intersection = base_results & sim_results
        union = base_results | sim_results
        jaccard_score = len(intersection) / len(union) if union else 0
        if jaccard_score > 0.8:
            local_similars.append({
                "similar_query": sim_query.strip(),
                "jaccard_score": round(jaccard_score, 3),
                "similar_results": list(sim_results)
            })
            if len(local_similars) >= max_similars:
                break
    return {
        "base_query": base_query.strip(),
        "base_results": list(base_results),
        "similar_queries": local_similars
    } if local_similars else None

In [ ]:
from functools import partial
import json
import multiprocessing
from itertools import combinations
from collections import defaultdict
from tqdm import tqdm

# Set your desired value for max_similars
max_similars = 20

# ---- Multiprocessing to compute similarities ----
cpu_count = multiprocessing.cpu_count()
args_list = [(i, q, r) for i, (idx, q, r) in enumerate(preprocessed)]
results = []
with multiprocessing.Pool(cpu_count) as pool:
    for result in tqdm(pool.imap_unordered(partial(process_query, max_similars=max_similars), args_list), total=len(args_list)):
        if result:
            results.append(result)

# ---- Save similarity results to JSON ----
with open("jaccard_similarities_optimized.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("JSON file written: jaccard_similarities_optimized.json")

In [ ]:
len(results)

In [ ]:
results[26564]["similar_queries"][0]["similar_query"]

In [ ]:
output = results[26564]["similar_queries"]

with open("similar_queries_output.txt", "w", encoding="utf-8") as f:
    if isinstance(output, (list, tuple)):
        for item in output:
            f.write(str(item) + "\n")
    else:
        f.write(str(output))


In [6]:
len(results)

2677

In [ ]:
import ijson
import json
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import torch
import numpy as np
from decimal import Decimal

# Detect GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"⚡ Using device: {device}")

# Load model on GPU if available
model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", device=device)

file_path = "jaccard_similarities_optimized.json"
output_file = "cosine_similarities_sorted.json"
count_empty = 0
results = []

# First pass: count items for tqdm progress
with open(file_path, "r", encoding="utf-8") as f:
    total_items = sum(1 for _ in ijson.items(f, "item"))

# Second pass: process with progress bar
with open(file_path, "r", encoding="utf-8") as f:
    for obj in tqdm(ijson.items(f, "item"), total=total_items, desc="Processing queries"):
        base_query = obj["base_query"]

        

        # Encode base query
        base_vec = model.encode(base_query, convert_to_numpy=True, device=device)

        # Collect all similar queries
        sim_queries = [sim_obj["similar_query"] for sim_obj in obj["similar_queries"]]

        if not sim_queries:
            count_empty += 1
            temp_query = base_query
        # Collect all similar queries
        sim_queries = [sim_obj["similar_query"] for sim_obj in obj["similar_queries"]]

        if sim_queries:  # only encode if non-empty
            sim_vecs = model.encode(sim_queries, convert_to_numpy=True, device=device, batch_size=32)
            sim_scores = cosine_similarity([base_vec], sim_vecs)[0]

            sim_queries_processed = []
            for sim_obj, sim_score in zip(obj["similar_queries"], sim_scores):
                sim_queries_processed.append({
                    "similar_query": sim_obj["similar_query"],
                    "jaccard_score": float(sim_obj["jaccard_score"]),
                    "cosine_score": float(sim_score)
                })

    # Sort by cosine_score descending
            sim_queries_processed.sort(key=lambda x: x["cosine_score"], reverse=True)
        else:
            sim_queries_processed = []
        results.append({
            "base_query": base_query,
            "similar_queries": sim_queries_processed
        })




⚡ Using device: cpu


Processing queries:  37%|███▋      | 9882/26565 [6:03:30<10:36:15,  2.29s/it]

In [10]:
# Save all results (ensure float conversion for JSON)
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f" Done! Saved {len(results)} processed queries to {output_file}")


✅ Done! Saved 0 processed queries to cosine_similarities_sorted.json


In [11]:
sim_queries_processed

[{'similar_query': 'PREFIX  rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>\nPREFIX  dbpo: <http://dbpedia.org/ontology/>\n\nSELECT DISTINCT  ?s\nWHERE\n  { ?s  rdf:type  dbpo:Book }',
  'jaccard_score': 1.0,
  'cosine_score': 0.8715481758117676}]